# PBIT Hyperparameter Tuning â€” LightGBM & XGBoost

Optuna search over hyperparameters using grouped 5-fold CV (StratifiedGroupKFold by Serial_Number).

## Studies in this notebook

| Study | Model | Objective | Trials | Config |
|-------|-------|-----------|-------:|--------|
| F1-tuned LightGBM | LightGBM | Maximise Macro F1 | 50 | Aug + FreqÃ—Sev |
| F1-tuned XGBoost  | XGBoost  | Maximise Macro F1 | 50 | Aug + FreqÃ—Sev |
| CMR-tuned LightGBM (Exp. A) | LightGBM | Minimise CMR (constraint: Bal Acc â‰¥ 0.50) | 100 | Aug + FreqÃ—Sev |

Secondary metrics tracked in all studies: F05 recall, F06 recall, Balanced Accuracy, Downward Risk.

## Setup

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

import lightgbm as lgb

from pbit.modeling import (
    EXPERIMENTS, META_COLS, FAULT_ORDER,
    make_cv_folds, run_cv_experiment, summarize_cv, results_table,
    compute_sample_weights, evaluate_predictions,
)
from pbit.data_validation import load_long_dataset
from pbit.augmentation import augment_training_set
from pbit.features import extract_features

TRAIN_REAL_PATH   = PROJECT_ROOT / 'data' / 'train_real.csv'
LONG_CLEAN_PATH   = PROJECT_ROOT / 'data' / 'PBIT_atuador_DESCARAC_r1_long_clean.csv'
BEST_PARAMS_PATH  = PROJECT_ROOT / 'data' / 'best_lgbm_params.json'
TUNED_RESULTS_PATH = PROJECT_ROOT / 'data' / 'cv_results_tuned.csv'

PROJECT_ROOT

## Load Data & Folds

In [ ]:
train_df = pd.read_csv(TRAIN_REAL_PATH)
long_df  = load_long_dataset(LONG_CLEAN_PATH)

feat_cols = [c for c in train_df.columns if c not in META_COLS]
folds     = make_cv_folds(train_df, n_splits=5, random_state=42)

print(f'Training set : {len(train_df):,} experiments x {len(feat_cols)} features')
print(f'Folds        : {len(folds)}')

## Optuna Objective

Each trial fits LightGBM across 5 CV folds with the `Aug + FreqÃ—Sev` configuration.  
A `f05_boost` multiplier (1â€“10Ã—) is applied on top of the frequencyÃ—severity weights  
for F05 samples only, giving the search explicit control over F05 recall.

In [ ]:
AUG_CFG = next(c for c in EXPERIMENTS if c.name == 'Aug + FreqÃ—Sev')

def objective(trial: optuna.Trial) -> float:
    params = {
        'n_estimators':      trial.suggest_int('n_estimators',      200, 800),
        'learning_rate':     trial.suggest_float('learning_rate',    0.01, 0.2, log=True),
        'num_leaves':        trial.suggest_int('num_leaves',         15,  127),
        'min_child_samples': trial.suggest_int('min_child_samples',  3,   50),
        'subsample':         trial.suggest_float('subsample',        0.5,  1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5,  1.0),
        'reg_alpha':         trial.suggest_float('reg_alpha',        1e-8, 10.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda',       1e-8, 10.0, log=True),
        'random_state':      42,
        'verbose':           -1,
    }
    f05_boost = trial.suggest_float('f05_boost', 1.0, 10.0)

    fold_f1s  = []
    fold_f05s = []
    fold_f06s = []

    for fold_i, (tr_idx, val_idx) in enumerate(folds):
        tr_real  = train_df.iloc[tr_idx]
        val_real = train_df.iloc[val_idx]

        # Augmentation (training fold only)
        tr_exp_ids = set(tr_real['Experiment_Index'].tolist())
        aug_long   = augment_training_set(
            long_df, tr_exp_ids,
            target_per_class=160, noise_scale=0.01,
            random_state=42 + fold_i,
        )
        if len(aug_long) > 0:
            aug_feat = extract_features(aug_long)
            tr_fold  = pd.concat([tr_real, aug_feat], ignore_index=True)
        else:
            tr_fold = tr_real

        X_train = tr_fold[feat_cols].values.astype(float)
        y_train = tr_fold['Fault_Mode'].values
        X_val   = val_real[feat_cols].values.astype(float)
        y_val   = val_real['Fault_Mode'].values

        # Freq x Sev weights + F05 boost
        sw = compute_sample_weights(y_train, 'freq_sev')
        sw[y_train == 'F05'] *= f05_boost

        model = lgb.LGBMClassifier(**params)
        model.fit(X_train, y_train, sample_weight=sw)
        y_pred = model.predict(X_val)

        metrics = evaluate_predictions(y_val, y_pred)
        fold_f1s.append(metrics['macro_f1'])
        fold_f05s.append(metrics['f05_recall'])
        fold_f06s.append(metrics['f06_recall'])

    mean_f1  = float(np.mean(fold_f1s))
    mean_f05 = float(np.nanmean([v for v in fold_f05s if not np.isnan(v)] or [0]))
    mean_f06 = float(np.nanmean([v for v in fold_f06s if not np.isnan(v)] or [0]))

    # Report secondary metrics as user attributes for inspection
    trial.set_user_attr('f05_recall', round(mean_f05, 4))
    trial.set_user_attr('f06_recall', round(mean_f06, 4))

    return mean_f1

## Run Optuna Study

50 trials with TPE sampler. Takes ~15â€“25 min.

In [ ]:
import time

sampler = optuna.samplers.TPESampler(seed=42)
study   = optuna.create_study(direction='maximize', sampler=sampler)

t0 = time.time()
study.optimize(objective, n_trials=50, show_progress_bar=True)
elapsed = time.time() - t0

print(f'\nCompleted {len(study.trials)} trials in {elapsed/60:.1f} min')
print(f'Best Macro F1 : {study.best_value:.4f}')
print(f'Best F05      : {study.best_trial.user_attrs["f05_recall"]}')
print(f'Best F06      : {study.best_trial.user_attrs["f06_recall"]}')
print(f'Best params   :')
for k, v in study.best_params.items():
    print(f'  {k}: {v}')

## Optimization History

In [ ]:
trials_df = study.trials_dataframe()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Optimization history
ax = axes[0]
ax.scatter(trials_df['number'], trials_df['value'], alpha=0.5, s=20, color='steelblue')
ax.plot(trials_df['number'],
        trials_df['value'].cummax(), color='tomato', linewidth=1.5, label='Best so far')
ax.set_xlabel('Trial')
ax.set_ylabel('Macro F1')
ax.set_title('Optimization history')
ax.legend()
ax.grid(True, linewidth=0.3)

# F05 recall vs Macro F1
ax = axes[1]
sc = ax.scatter(
    trials_df['value'],
    trials_df['user_attrs_f05_recall'],
    c=trials_df['user_attrs_f06_recall'],
    cmap='RdYlGn', alpha=0.7, s=25,
)
plt.colorbar(sc, ax=ax, label='F06 recall')
ax.set_xlabel('Macro F1')
ax.set_ylabel('F05 recall')
ax.set_title('F05 recall vs Macro F1 (colour = F06 recall)')
ax.grid(True, linewidth=0.3)

# Mark best trial
bt = study.best_trial
ax.scatter(
    bt.value, bt.user_attrs['f05_recall'],
    marker='*', s=200, color='gold', edgecolors='black', zorder=5, label='Best'
)
ax.legend()

plt.tight_layout()
plt.show()

## Parameter Importance

In [ ]:
importance = optuna.importance.get_param_importances(study)

fig, ax = plt.subplots(figsize=(8, 4))
keys   = list(importance.keys())
vals   = list(importance.values())
colors = ['tomato' if k == 'f05_boost' else 'steelblue' for k in keys]
ax.barh(keys[::-1], vals[::-1], color=colors[::-1])
ax.set_xlabel('Importance')
ax.set_title('Hyperparameter importance (fANOVA)')
ax.grid(True, axis='x', linewidth=0.3)
plt.tight_layout()
plt.show()

print('\nParameter importances:')
for k, v in importance.items():
    print(f'  {k:<22}: {v:.4f}')

## Top 10 Trials

In [ ]:
top10 = (
    trials_df
    .sort_values('value', ascending=False)
    .head(10)
    [['number', 'value', 'user_attrs_f05_recall', 'user_attrs_f06_recall']]
    .rename(columns={
        'number': 'Trial',
        'value':  'Macro_F1',
        'user_attrs_f05_recall': 'F05_recall',
        'user_attrs_f06_recall': 'F06_recall',
    })
    .reset_index(drop=True)
)
top10

## Save Best Parameters

In [ ]:
best = study.best_params.copy()
best['random_state'] = 42
best['verbose']      = -1

with open(BEST_PARAMS_PATH, 'w') as f:
    json.dump(best, f, indent=2)

print(f'Saved: {BEST_PARAMS_PATH}')
print(json.dumps(best, indent=2))

## Re-run All 6 Configs with Best Params

The `f05_boost` parameter found by Optuna is applied to all configs that use sample weights.  
Configs with `weight_mode='none'` are run without extra boost.

In [ ]:
import time
from pbit.modeling import ExperimentConfig

f05_boost_val = study.best_params['f05_boost']

# Extract only LightGBM params (exclude f05_boost)
lgbm_params = {k: v for k, v in study.best_params.items() if k != 'f05_boost'}
lgbm_params['random_state'] = 42
lgbm_params['verbose']      = -1

print(f'f05_boost     : {f05_boost_val:.2f}')
print(f'LightGBM params: {lgbm_params}')
print()

In [ ]:
def run_cv_with_f05_boost(
    cfg: ExperimentConfig,
    lgbm_params: dict,
    f05_boost: float,
    train_df: pd.DataFrame,
    long_df,
    feat_cols: list,
    folds: list,
    aug_target: int = 160,
) -> list:
    """Like run_cv_experiment but injects F05 sample weight boost."""
    fold_results = []
    for fold_i, (tr_idx, val_idx) in enumerate(folds):
        tr_real  = train_df.iloc[tr_idx]
        val_real = train_df.iloc[val_idx]

        if cfg.augment:
            tr_exp_ids = set(tr_real['Experiment_Index'].tolist())
            aug_long   = augment_training_set(
                long_df, tr_exp_ids,
                target_per_class=aug_target, noise_scale=0.01,
                random_state=42 + fold_i,
            )
            if len(aug_long) > 0:
                aug_feat = extract_features(aug_long)
                tr_fold  = pd.concat([tr_real, aug_feat], ignore_index=True)
            else:
                tr_fold = tr_real
        else:
            tr_fold = tr_real

        X_train = tr_fold[feat_cols].values.astype(float)
        y_train = tr_fold['Fault_Mode'].values
        X_val   = val_real[feat_cols].values.astype(float)
        y_val   = val_real['Fault_Mode'].values

        sw = compute_sample_weights(y_train, cfg.weight_mode)
        if f05_boost > 1.0 and cfg.weight_mode != 'none':
            sw[y_train == 'F05'] *= f05_boost

        model = lgb.LGBMClassifier(**lgbm_params)
        model.fit(X_train, y_train, sample_weight=sw)
        y_pred = model.predict(X_val)

        metrics = evaluate_predictions(y_val, y_pred)
        metrics['fold'] = fold_i
        fold_results.append(metrics)
    return fold_results


tuned_summaries = {}
baseline_summaries = {}  # reload from nb04 results for comparison

t0 = time.time()
for cfg in EXPERIMENTS:
    label = f'Tuned / {cfg.name}'
    fold_res = run_cv_with_f05_boost(
        cfg, lgbm_params, f05_boost_val,
        train_df, long_df, feat_cols, folds,
    )
    summary = summarize_cv(fold_res)
    tuned_summaries[label] = summary
    f1  = summary.get('macro_f1_mean', float('nan'))
    f06 = summary.get('f06_recall_mean', float('nan'))
    f05 = summary.get('f05_recall_mean', float('nan'))
    print(f'{label:<38}  F1={f1:.3f}  F06={f06:.3f}  F05={f05:.3f}')

print(f'\nDone in {time.time()-t0:.0f}s')

## Tuned Results Table

In [ ]:
tuned_tbl = results_table(tuned_summaries)
tuned_tbl

## Comparison: Baseline LightGBM vs Tuned LightGBM

In [ ]:
baseline_csv = pd.read_csv(PROJECT_ROOT / 'data' / 'cv_results.csv')
baseline_lgb = baseline_csv[baseline_csv['Experiment'].str.startswith('LightGBM')].copy()
baseline_lgb['Config'] = baseline_lgb['Experiment'].str.replace('LightGBM / ', '', regex=False)
baseline_lgb['Source'] = 'Baseline'

tuned_rows = []
for label, s in tuned_summaries.items():
    cfg_name = label.replace('Tuned / ', '')
    tuned_rows.append({
        'Config':      cfg_name,
        'Source':      'Tuned',
        'Macro_F1':    round(s.get('macro_f1_mean', float('nan')), 4),
        'F06_Recall':  round(s.get('f06_recall_mean', float('nan')), 4),
        'F05_Recall':  round(s.get('f05_recall_mean', float('nan')), 4),
        'F04_Recall':  round(s.get('f04_recall_mean', float('nan')), 4),
        'Sev3_Recall': round(s.get('sev3_recall_mean', float('nan')), 4),
    })
tuned_comp = pd.DataFrame(tuned_rows)

comp = pd.concat([
    baseline_lgb[['Config', 'Source', 'Macro_F1', 'F06_Recall', 'F05_Recall', 'F04_Recall', 'Sev3_Recall']],
    tuned_comp,
], ignore_index=True).sort_values(['Config', 'Source'])

comp

## Gain Chart: Tuned vs Baseline

In [ ]:
metrics_to_plot = ['Macro_F1', 'F05_Recall', 'F06_Recall', 'F04_Recall']
configs = [c.name for c in EXPERIMENTS]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for ax, metric in zip(axes, metrics_to_plot):
    base_vals  = []
    tuned_vals = []
    for cfg_name in configs:
        b = comp[(comp['Config'] == cfg_name) & (comp['Source'] == 'Baseline')][metric]
        t = comp[(comp['Config'] == cfg_name) & (comp['Source'] == 'Tuned')][metric]
        base_vals.append(float(b.values[0]) if len(b) else float('nan'))
        tuned_vals.append(float(t.values[0]) if len(t) else float('nan'))

    x = np.arange(len(configs))
    w = 0.35
    ax.bar(x - w/2, base_vals,  w, label='Baseline', color='steelblue', alpha=0.8)
    ax.bar(x + w/2, tuned_vals, w, label='Tuned',    color='tomato',    alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(configs, rotation=30, ha='right', fontsize=8)
    ax.set_title(metric)
    ax.legend(fontsize=8)
    ax.grid(True, axis='y', linewidth=0.3)
    ax.set_ylim(0, 1)

fig.suptitle('LightGBM: Baseline vs Tuned (Macro F1, F04/F05/F06 recall)', fontsize=12)
plt.tight_layout()
plt.show()

## Per-Class Recall Heatmap â€” Tuned

In [ ]:
rows = []
for label, s in tuned_summaries.items():
    row = {'Experiment': label}
    for cls in FAULT_ORDER:
        row[cls] = round(s['per_class_recall_mean'].get(cls, float('nan')), 3)
    rows.append(row)

heatmap_df = pd.DataFrame(rows).set_index('Experiment')

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(
    heatmap_df.astype(float), annot=True, fmt='.2f',
    cmap='RdYlGn', vmin=0, vmax=1,
    linewidths=0.4, ax=ax,
)
ax.set_title('Tuned LightGBM â€” Mean recall per class (5-fold CV)', fontsize=12)
ax.set_xlabel('Fault Mode')
plt.tight_layout()
plt.show()

## Save Tuned Results

In [ ]:
tuned_tbl.to_csv(TUNED_RESULTS_PATH, index=False)
print(f'Saved: {TUNED_RESULTS_PATH}')
tuned_tbl

---
## XGBoost Hyperparameter Optimisation

### Objective Function

In [ ]:
from pbit.modeling import XGBWrapper

def xgb_objective(trial: optuna.Trial) -> float:
    params = {
        'n_estimators':    trial.suggest_int('n_estimators',    200, 800),
        'learning_rate':   trial.suggest_float('learning_rate',  0.01, 0.2, log=True),
        'max_depth':       trial.suggest_int('max_depth',        3,   10),
        'min_child_weight':trial.suggest_int('min_child_weight', 1,   20),
        'subsample':       trial.suggest_float('subsample',      0.5,  1.0),
        'colsample_bytree':trial.suggest_float('colsample_bytree',0.5, 1.0),
        'reg_alpha':       trial.suggest_float('reg_alpha',      1e-8, 10.0, log=True),
        'reg_lambda':      trial.suggest_float('reg_lambda',     1e-8, 10.0, log=True),
        'eval_metric':     'mlogloss',
        'random_state':    42,
        'verbosity':       0,
    }
    f05_boost = trial.suggest_float('f05_boost', 1.0, 10.0)

    fold_f1s, fold_f05s, fold_f06s = [], [], []

    for fold_i, (tr_idx, val_idx) in enumerate(folds):
        tr_real  = train_df.iloc[tr_idx]
        val_real = train_df.iloc[val_idx]

        tr_exp_ids = set(tr_real['Experiment_Index'].tolist())
        aug_long   = augment_training_set(
            long_df, tr_exp_ids,
            target_per_class=160, noise_scale=0.01,
            random_state=42 + fold_i,
        )
        if len(aug_long) > 0:
            aug_feat = extract_features(aug_long)
            tr_fold  = pd.concat([tr_real, aug_feat], ignore_index=True)
        else:
            tr_fold = tr_real

        X_train = tr_fold[feat_cols].values.astype(float)
        y_train = tr_fold['Fault_Mode'].values
        X_val   = val_real[feat_cols].values.astype(float)
        y_val   = val_real['Fault_Mode'].values

        sw = compute_sample_weights(y_train, 'freq_sev')
        sw[y_train == 'F05'] *= f05_boost

        m = XGBWrapper(**params)
        m.fit(X_train, y_train, sample_weight=sw)
        y_pred = m.predict(X_val)

        metrics = evaluate_predictions(y_val, y_pred)
        fold_f1s.append(metrics['macro_f1'])
        fold_f05s.append(metrics['f05_recall'])
        fold_f06s.append(metrics['f06_recall'])

    mean_f1  = float(np.mean(fold_f1s))
    mean_f05 = float(np.nanmean([v for v in fold_f05s if not np.isnan(v)] or [0]))
    mean_f06 = float(np.nanmean([v for v in fold_f06s if not np.isnan(v)] or [0]))
    trial.set_user_attr('f05_recall', round(mean_f05, 4))
    trial.set_user_attr('f06_recall', round(mean_f06, 4))
    return mean_f1

In [ ]:
xgb_sampler = optuna.samplers.TPESampler(seed=42)
xgb_study   = optuna.create_study(direction='maximize', sampler=xgb_sampler)

t0 = time.time()
xgb_study.optimize(xgb_objective, n_trials=50, show_progress_bar=True)
elapsed = time.time() - t0

bt = xgb_study.best_trial
print(f'Completed {len(xgb_study.trials)} trials in {elapsed/60:.1f} min')
print(f'Best Macro F1 : {xgb_study.best_value:.4f}')
print('Best F05      :', bt.user_attrs['f05_recall'])
print('Best F06      :', bt.user_attrs['f06_recall'])
print('Best params:')
for k, v in xgb_study.best_params.items():
    print(f'  {k}: {v}')

### Optimisation History â€” XGBoost

In [ ]:
xgb_trials_df = xgb_study.trials_dataframe()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.scatter(xgb_trials_df['number'], xgb_trials_df['value'], alpha=0.5, s=20, color='darkorange')
ax.plot(xgb_trials_df['number'], xgb_trials_df['value'].cummax(),
        color='tomato', linewidth=1.5, label='Best so far')
ax.set_xlabel('Trial'); ax.set_ylabel('Macro F1')
ax.set_title('XGBoost optimisation history')
ax.legend(); ax.grid(True, linewidth=0.3)

ax = axes[1]
sc = ax.scatter(
    xgb_trials_df['value'],
    xgb_trials_df['user_attrs_f05_recall'],
    c=xgb_trials_df['user_attrs_f06_recall'],
    cmap='RdYlGn', alpha=0.7, s=25,
)
plt.colorbar(sc, ax=ax, label='F06 recall')
ax.set_xlabel('Macro F1'); ax.set_ylabel('F05 recall')
ax.set_title('XGBoost: F05 vs Macro F1')
ax.grid(True, linewidth=0.3)
bx = xgb_study.best_trial
ax.scatter(bx.value, bx.user_attrs['f05_recall'],
           marker='*', s=200, color='gold', edgecolors='black', zorder=5, label='Best')
ax.legend()
plt.tight_layout(); plt.show()

xgb_importance = optuna.importance.get_param_importances(xgb_study)
print('XGBoost parameter importances:')
for k, v in xgb_importance.items():
    print(f'  {k:<22}: {v:.4f}')

### Top 10 XGBoost Trials

In [ ]:
xgb_top10 = (
    xgb_trials_df
    .sort_values('value', ascending=False)
    .head(10)
    [['number', 'value', 'user_attrs_f05_recall', 'user_attrs_f06_recall']]
    .rename(columns={'number':'Trial','value':'Macro_F1',
                     'user_attrs_f05_recall':'F05_recall',
                     'user_attrs_f06_recall':'F06_recall'})
    .reset_index(drop=True)
)
xgb_top10

### Save Best XGBoost Parameters

In [ ]:
XGB_PARAMS_PATH = PROJECT_ROOT / 'data' / 'best_xgb_params.json'

best_xgb = xgb_study.best_params.copy()
best_xgb['eval_metric']  = 'mlogloss'
best_xgb['random_state'] = 42
best_xgb['verbosity']    = 0

with open(XGB_PARAMS_PATH, 'w') as f:
    json.dump(best_xgb, f, indent=2)

print('Saved:', XGB_PARAMS_PATH)
print(json.dumps(best_xgb, indent=2))

### Re-run 6 Configs â€” Tuned XGBoost

In [ ]:
xgb_f05_boost = xgb_study.best_params['f05_boost']
xgb_params    = {k: v for k, v in xgb_study.best_params.items() if k != 'f05_boost'}
xgb_params.update({'eval_metric':'mlogloss','random_state':42,'verbosity':0})

xgb_tuned_summaries = {}

for cfg in EXPERIMENTS:
    label = f'XGB-Tuned / {cfg.name}'
    fold_results = []
    for fold_i, (tr_idx, val_idx) in enumerate(folds):
        tr_real  = train_df.iloc[tr_idx]
        val_real = train_df.iloc[val_idx]
        if cfg.augment:
            tr_exp_ids = set(tr_real['Experiment_Index'].tolist())
            aug_long   = augment_training_set(
                long_df, tr_exp_ids, target_per_class=160,
                noise_scale=0.01, random_state=42+fold_i)
            if len(aug_long) > 0:
                aug_feat = extract_features(aug_long)
                tr_fold  = pd.concat([tr_real, aug_feat], ignore_index=True)
            else:
                tr_fold = tr_real
        else:
            tr_fold = tr_real
        X_train = tr_fold[feat_cols].values.astype(float)
        y_train = tr_fold['Fault_Mode'].values
        X_val   = val_real[feat_cols].values.astype(float)
        y_val   = val_real['Fault_Mode'].values
        sw = compute_sample_weights(y_train, cfg.weight_mode)
        if xgb_f05_boost > 1.0 and cfg.weight_mode != 'none':
            sw[y_train == 'F05'] *= xgb_f05_boost
        m = XGBWrapper(**xgb_params)
        m.fit(X_train, y_train, sample_weight=sw)
        metrics = evaluate_predictions(y_val, m.predict(X_val))
        metrics['fold'] = fold_i
        fold_results.append(metrics)
    summary = summarize_cv(fold_results)
    xgb_tuned_summaries[label] = summary
    f1  = summary.get('macro_f1_mean', float('nan'))
    f06 = summary.get('f06_recall_mean', float('nan'))
    f05 = summary.get('f05_recall_mean', float('nan'))
    print(f'{label:<40}  F1={f1:.3f}  F06={f06:.3f}  F05={f05:.3f}')

---
## Final Comparison: Tuned LightGBM vs Tuned XGBoost

In [ ]:
lgb_rows, xgb_rows = [], []
for cfg in EXPERIMENTS:
    cfg_name = cfg.name
    lgb_s = tuned_summaries.get(f'Tuned / {cfg_name}', {})
    xgb_s = xgb_tuned_summaries.get(f'XGB-Tuned / {cfg_name}', {})
    lgb_rows.append({'Config': cfg_name, 'Model': 'LightGBM',
        'Macro_F1':   round(lgb_s.get('macro_f1_mean', float('nan')), 4),
        'F06_Recall': round(lgb_s.get('f06_recall_mean', float('nan')), 4),
        'F05_Recall': round(lgb_s.get('f05_recall_mean', float('nan')), 4),
        'F04_Recall': round(lgb_s.get('f04_recall_mean', float('nan')), 4),
        'Sev3_Recall':round(lgb_s.get('sev3_recall_mean',float('nan')), 4),
    })
    xgb_rows.append({'Config': cfg_name, 'Model': 'XGBoost',
        'Macro_F1':   round(xgb_s.get('macro_f1_mean', float('nan')), 4),
        'F06_Recall': round(xgb_s.get('f06_recall_mean', float('nan')), 4),
        'F05_Recall': round(xgb_s.get('f05_recall_mean', float('nan')), 4),
        'F04_Recall': round(xgb_s.get('f04_recall_mean', float('nan')), 4),
        'Sev3_Recall':round(xgb_s.get('sev3_recall_mean',float('nan')), 4),
    })

cmp_df = pd.DataFrame(lgb_rows + xgb_rows).sort_values(['Config','Model']).reset_index(drop=True)
cmp_df

### Best Config per Model

In [ ]:
best_lgb = cmp_df[cmp_df['Model']=='LightGBM'].sort_values('Macro_F1',ascending=False).iloc[0]
best_xgb = cmp_df[cmp_df['Model']=='XGBoost'].sort_values('Macro_F1',ascending=False).iloc[0]

print('Best LightGBM config :', best_lgb['Config'])
print('  Macro F1 :', best_lgb['Macro_F1'], ' F06:', best_lgb['F06_Recall'],
      ' F05:', best_lgb['F05_Recall'], ' Sev3:', best_lgb['Sev3_Recall'])
print()
print('Best XGBoost config  :', best_xgb['Config'])
print('  Macro F1 :', best_xgb['Macro_F1'], ' F06:', best_xgb['F06_Recall'],
      ' F05:', best_xgb['F05_Recall'], ' Sev3:', best_xgb['Sev3_Recall'])

metrics_cols = ['Macro_F1','F06_Recall','F05_Recall','F04_Recall']
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, col in zip(axes, metrics_cols):
    lgb_vals = cmp_df[cmp_df['Model']=='LightGBM'][col].values
    xgb_vals = cmp_df[cmp_df['Model']=='XGBoost'][col].values
    x = np.arange(len(EXPERIMENTS))
    ax.bar(x-0.2, lgb_vals, 0.4, label='LightGBM', color='steelblue', alpha=0.85)
    ax.bar(x+0.2, xgb_vals, 0.4, label='XGBoost',  color='darkorange', alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels([c.name[:8] for c in EXPERIMENTS], rotation=30, ha='right', fontsize=7)
    ax.set_title(col, fontsize=10); ax.set_ylim(0, 1)
    ax.legend(fontsize=7); ax.grid(True, axis='y', linewidth=0.3)
fig.suptitle('Tuned LightGBM vs Tuned XGBoost â€” all configs', fontsize=11)
plt.tight_layout(); plt.show()

### Save XGBoost Tuned Results

In [ ]:
XGB_TUNED_PATH = PROJECT_ROOT / 'data' / 'cv_results_xgb_tuned.csv'
xgb_tbl = results_table(xgb_tuned_summaries)
xgb_tbl.to_csv(XGB_TUNED_PATH, index=False)
print('Saved:', XGB_TUNED_PATH)
xgb_tbl